# OpenRFM quickstart

Predict `rel-f1/driver-dnf` with recipe R2 (frozen backbone, memory read, record bank and the frozen Kumo-Tabular
consumer) and print the TEST AUROC. Use a GPU runtime.

Clone the repository.

In [ ]:
import os, subprocess
if not os.path.isdir('openrfm'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Quant-Spirit/openrfm', 'openrfm'], check=True)

Install the package and the archive converter; the converter is built from Rust source.

In [ ]:
!command -v cargo || (curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal)
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!pip install -q ./openrfm ./openrfm/native/rustler

Download the RelBench database `rel-f1` and its tasks from the official release (checked by sha256). `run` does
this by itself when the database is absent.

In [ ]:
from pathlib import Path
from openrfm.infer.download import download

DATA = Path('relbench')
print(download('rel-f1', DATA))

The two checkpoints come from the Hugging Face Hub (`zkchen/openrfm`) and are checked by sha256.

In [ ]:
from openrfm import hub
print(hub.checkpoint('backbone'), hub.checkpoint('memory'))

Convert, sample contexts, encode and predict. The first run converts the database and builds the sampler kernels.

In [ ]:
from openrfm.infer.run import run

result = run('rel-f1/driver-dnf', 'r2', DATA, 'openrfm-work')
print(f"TEST {result['metric']} {result['value']:.4f} +- {result['SE']:.4f}")

Your own database: `make_shop.py` writes a simulated shop database in RelBench layout, and a short YAML file
declares its task. Every TEST row is predicted with the recipe's default cell and scored against its label.

In [ ]:
!python openrfm/examples/private/make_shop.py --out {DATA}
shop = run(None, 'r1', DATA, 'openrfm-work', spec='openrfm/examples/private/customer-churn.yaml')
print(f"shop customer-churn TEST {shop['metric']} {shop['value']:.4f} +- {shop['SE']:.4f}")